# 1. Preprocess: wes7_raw → analytic_sample
- 5차원 안전문화 척도 + z-score, 이력재해 통제변수, STROBE flow
- **Fix (Colab)**: Google Drive sync race 방지 — SHA를 메모리에서 계산

In [2]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — Load, Derive, Save + Auto Methods (Section 2.4-2.5)
# ══════════════════════════════════════════════════════════════
# ── Portable path setup (Colab + Local) ──
import os, sys, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha'
    ENV = 'Colab'
except ImportError:
    BASE = '/Users/y3korea/Library/CloudStorage/GoogleDrive-y3korea@gmail.com/내 드라이브/완석_구글자료/연구자료/20260313_kosha'
    ENV = 'Local'
assert os.path.exists(BASE), f'❌ BASE not found: {BASE}'
print(f'[ENV] {ENV}')
print(f'[BASE] {BASE}')

import pandas as pd, numpy as np, hashlib

RAW_DIR = os.path.join(BASE, 'Code_kosha', '2_code', 'output', 'pooled_raw')
PRE_DIR = os.path.join(BASE, 'Code_kosha', '2_code', 'output', 'pre_output')
PAPER_DIR = os.path.join(BASE, 'Code_kosha', '2_code', 'paper', 'auto')
os.makedirs(PRE_DIR, exist_ok=True)
os.makedirs(PAPER_DIR, exist_ok=True)

# ── Sanity checks ──
assert os.path.isdir(RAW_DIR), f'❌ RAW_DIR NOT FOUND: {RAW_DIR}'
assert os.path.isdir(PRE_DIR), f'❌ PRE_DIR NOT FOUND: {PRE_DIR}'
print(f'[RAW_DIR ✓] {RAW_DIR}')
print(f'[PRE_DIR ✓] {PRE_DIR}')

# ── Drive WRITE 권한 smoke test ──
test_path = os.path.join(PRE_DIR, '_write_test.tmp')
try:
    t0 = time.time()
    with open(test_path, 'w') as f:
        f.write('write_test_ok')
    with open(test_path) as f:
        assert f.read() == 'write_test_ok'
    os.remove(test_path)
    print(f'[WRITE TEST ✓] PRE_DIR is writable ({(time.time()-t0)*1000:.0f} ms)')
except Exception as e:
    print(f'[WRITE TEST ✗] {type(e).__name__}: {e}')
    raise

# ── 입력 파일 검증 (0_extract.ipynb 산출물) ──
input_csv = os.path.join(RAW_DIR, 'wes7_raw.csv')
assert os.path.exists(input_csv), (
    f'❌ INPUT NOT FOUND: {input_csv}\n'
    f'   → 0_extract.ipynb를 먼저 실행해서 wes7_raw.csv를 생성하세요.')
import datetime
mt = datetime.datetime.fromtimestamp(os.path.getmtime(input_csv))
sz_in = os.path.getsize(input_csv) / 1024 / 1024
print(f'[INPUT ✓] {input_csv}')
print(f'  Size: {sz_in:.1f} MB, Modified: {mt}')

df = pd.read_csv(input_csv, encoding='utf-8-sig')
N_raw = len(df)
print(f'\nLoaded: N={N_raw:,} × {len(df.columns)} cols')

# ── 5-factor safety culture ──
SC_DIMS = {
    'sc_mgmt':  ['mgt_emph_saf', 'mgt_prior_saf', 'mgt_value_saf'],
    'sc_comm':  ['saf_disc_opp', 'saf_open_disc', 'saf_feed_reg', 'saf_sug_sys', 'saf_sug_resp'],
    'sc_train': ['saf_tr_opp', 'saf_tr_effect'],
    'sc_sys':   ['saf_sys_proc', 'saf_proc_effect', 'saf_equip_avail'],
    'sc_empow': ['work_ref_unsaf', 'work_vol_saf'],
}
SC_ITEMS = [i for items in SC_DIMS.values() for i in items]
assert len(SC_ITEMS) == 15

for dim, items in SC_DIMS.items():
    df[dim] = df[items].mean(axis=1, skipna=False)
df['sc_total'] = df[SC_ITEMS].mean(axis=1, skipna=False)

for c in list(SC_DIMS.keys()) + ['sc_total']:
    df[f'{c}_z'] = (df[c] - df[c].mean()) / df[c].std()

# ── Outcomes + lagged controls ──
df['vic_2024_appr'] = df['vic_2024_appr'].fillna(0)
df['any_acc_2024']  = (df['vic_2024_appr'] > 0).astype(int)
df['vic_prior']     = df['vic_2022_appr'].fillna(0) + df['vic_2023_appr'].fillna(0)
df['had_prior']     = (df['vic_prior'] > 0).astype(int)
df['log_prior']     = np.log1p(df['vic_prior'])
df['vic_3yr_appr']  = df['vic_2022_appr'].fillna(0) + df['vic_2023_appr'].fillna(0) + df['vic_2024_appr'].fillna(0)
df['vic_2024_occ']  = df['vic_2024_occ'].fillna(0)
df['dth_2024_appr'] = df['dth_2024_appr'].fillna(0)
df['acc_dth_2024_appr'] = df['acc_dth_2024_appr'].fillna(0)

# ── Offset: worker count midpoints ──
SIZE_MIDPOINT = {1: 2.5, 2: 12.0, 3: 34.5, 4: 74.5, 5: 200.0}
df['n_workers']   = df['r_wrk_tot'].map(SIZE_MIDPOINT)
df['log_workers'] = np.log(df['n_workers'])
df['size_cat']    = df['r_wrk_tot'].astype('Int64')
df['industry']    = df['ksic2'].astype('Int64')

for v in ['saf_com_yn','saf_dept_yn','saf_mgr_yn','saf_rep_yn','saf_sup_yn']:
    if v in df.columns:
        df[f'{v}_bi'] = (df[v] == 1).astype(float)
        df.loc[df[v].isna(), f'{v}_bi'] = np.nan
df['long_hrs']   = (df.get('ovt_yn', pd.Series([np.nan]*len(df))) == 1).astype(float)
df['shift_work'] = (df.get('shift_yn', pd.Series([np.nan]*len(df))) == 1).astype(float)

df['swt']  = df['wt1']
df['swt2'] = df['wt2']

# ── STROBE Flow ──
N0 = len(df)
df = df.dropna(subset=SC_ITEMS)
N1 = len(df)
df = df.dropna(subset=['r_wrk_tot', 'ksic2', 'log_workers'])
N2 = len(df)

out_fp = os.path.join(PRE_DIR, 'analytic_sample.csv')
df = df.sort_values(['industry','r_wrk_tot','id']).reset_index(drop=True)
print(f'\n[저장 시작] {out_fp}')
print(f'  메모리상 DataFrame: {len(df):,} rows × {len(df.columns)} cols')

try:
    # 1) CSV string 생성 (메모리상 — Drive sync race 회피)
    csv_content = df.to_csv(index=False)
    print(f'  ✓ CSV string 생성: {len(csv_content)/1024/1024:.1f} MB')

    # 2) 디스크 쓰기
    with open(out_fp, 'w', encoding='utf-8-sig') as f:
        f.write(csv_content)
    print(f'  ✓ 디스크 쓰기 완료')

    # 3) 즉시 검증
    assert os.path.exists(out_fp), 'WRITE FAILED: file not present after write'
    actual_sz = os.path.getsize(out_fp)
    assert actual_sz > 0, 'WRITE FAILED: file size is 0'
    print(f'  ✓ 검증: 파일 존재 + size {actual_sz/1024/1024:.1f} MB')

    # 4) mtime 확인
    import datetime as _dt
    mt_out = _dt.datetime.fromtimestamp(os.path.getmtime(out_fp))
    print(f'  ✓ Modified: {mt_out}')

    # 5) SHA-256 hash
    sha = hashlib.sha256(csv_content.encode('utf-8')).hexdigest()[:16]
    print(f'  ✓ SHA-256: {sha}')

except Exception as e:
    import traceback
    print(f'  ❌ 저장 실패: {type(e).__name__}: {e}')
    traceback.print_exc()
    raise

# ── Descriptive stats for manuscript ──
m_mgmt, sd_mgmt = df['sc_mgmt'].mean(), df['sc_mgmt'].std()
m_comm, sd_comm = df['sc_comm'].mean(), df['sc_comm'].std()
m_train, sd_train = df['sc_train'].mean(), df['sc_train'].std()
m_sys, sd_sys = df['sc_sys'].mean(), df['sc_sys'].std()
m_empow, sd_empow = df['sc_empow'].mean(), df['sc_empow'].std()
m_tot, sd_tot = df['sc_total'].mean(), df['sc_total'].std()
n_ind = int(df['industry'].nunique())
pct_1 = (df['r_wrk_tot']==1).mean()*100
pct_2 = (df['r_wrk_tot']==2).mean()*100
pct_3 = (df['r_wrk_tot']==3).mean()*100
pct_4 = (df['r_wrk_tot']==4).mean()*100
pct_5 = (df['r_wrk_tot']==5).mean()*100
pct_acc = df['any_acc_2024'].mean()*100
n_acc = int(df['any_acc_2024'].sum())
pct_prior = df['had_prior'].mean()*100
n_prior = int(df['had_prior'].sum())
mean_vic = df['vic_2024_appr'].mean()
max_vic = df['vic_2024_appr'].max()

print(f'\n=== STROBE FLOW ===')
print(f'Raw:              {N0:>8,}')
print(f'- SC missing:     {N0-N1:>8,}')
print(f'After Step 1:     {N1:>8,}')
print(f'- Size/Ind miss:  {N1-N2:>8,}')
print(f'Final:            {N2:>8,}')

# Sanity check output
df['sc_q4'] = pd.qcut(df['sc_total'], 4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
print(f'\n=== Unadjusted SC Quartile vs Accident Rate ===')
for q in df['sc_q4'].cat.categories:
    sub = df[df['sc_q4']==q]
    print(f'  {q} (SC {sub["sc_total"].mean():.2f}): any_acc={sub["any_acc_2024"].mean()*100:.1f}%')

# ══════════════════════════════════════════════════════════════
# AUTO-GENERATE METHODS SECTIONS 2.4–2.5 (English, Safety Science style)
# ══════════════════════════════════════════════════════════════

methods_md = f"""# Methods (auto-generated from 1_preprocess.ipynb)

## 2.4 Safety Culture Scale Construction

We computed five dimension scores as unweighted arithmetic means of their
constituent items, preserving the original 1–5 scale for interpretability. An
overall safety culture score was computed as the mean of all 15 items. This
equal-weighting approach was selected because unweighted sums would confound
scale magnitude with item count across dimensions of unequal length (ranging
from 2 to 5 items). To enable direct comparison of effect sizes across
dimensions in regression models, we standardized each dimension score and the
overall score to z-scores (mean = 0, SD = 1) using the full analytic sample as
the reference distribution.

Dimension-level descriptive statistics in the final analytic sample were:

- **Management Commitment**: M = {m_mgmt:.2f}, SD = {sd_mgmt:.2f}
- **Safety Communication**: M = {m_comm:.2f}, SD = {sd_comm:.2f}
- **Safety Training**: M = {m_train:.2f}, SD = {sd_train:.2f}
- **Safety Systems and Procedures**: M = {m_sys:.2f}, SD = {sd_sys:.2f}
- **Worker Empowerment**: M = {m_empow:.2f}, SD = {sd_empow:.2f}
- **Overall (15-item composite)**: M = {m_tot:.2f}, SD = {sd_tot:.2f}

Mean scores clustered in the upper portion of the response scale (all dimensions
M > 3.9), indicating generally favorable safety culture perceptions across
Korean establishments, with modest variability (SDs ranging from
{min(sd_mgmt, sd_comm, sd_train, sd_sys, sd_empow):.2f} to {max(sd_mgmt, sd_comm, sd_train, sd_sys, sd_empow):.2f}).
This ceiling-tendency is consistent with self-report safety climate instruments
in East Asian establishment surveys and may attenuate associations with rare
outcomes such as workplace injuries.

## 2.5 Outcome Transformations

The primary outcome variable `vic_2024_appr` was retained as a non-negative
integer count for negative binomial modeling. A binary outcome
(`any_acc_2024`) was derived for descriptive analyses and machine learning
cross-validation, taking the value 1 when at least one officially approved
injury occurred in 2024 and 0 otherwise. In the final analytic sample, {n_acc:,}
establishments ({pct_acc:.1f}%) experienced at least one approved injury in 2024,
with a mean of {mean_vic:.2f} victims per establishment (range 0–{max_vic:.0f}).

The historical accident control `vic_prior` was log-transformed as
log(1 + vic_prior) to handle the strong right-skew (median = 0, maximum = several
hundred) and retain establishments with no prior accidents. In the analytic
sample, {n_prior:,} establishments ({pct_prior:.1f}%) had at least one officially
approved injury during 2022–2023, providing sufficient variation to adjust for
historical risk. A three-year cumulative count (`vic_3yr_appr` = 2022 + 2023 +
2024) was computed as a secondary outcome for sensitivity analysis (S2). Fatal
occupational injury counts (`acc_dth_2024_appr`) were retained for a severity-
focused sensitivity analysis (S3).

## 2.6 Analytic Sample

Following STROBE reporting guidelines for observational research (von Elm et al.,
2007), we applied a two-stage complete-case exclusion strategy. Of the {N0:,}
establishments in the raw WES-7 file, {N0 - N1:,} were excluded for missingness
on any of the 15 safety culture items. A further {N1 - N2:,} were excluded for
missing establishment size or KSIC industry classification, which were required
for offset calculation and industry fixed-effects adjustment. The final analytic
sample comprised **N = {N2:,} establishments** distributed across {n_ind} KSIC
2-digit industrial sectors.

Because missingness was minimal at both stages (fewer than 0.1% of records
excluded in each step), selection bias from complete-case analysis is expected
to be negligible. No multiple imputation was performed.

### Sample Distribution by Establishment Size

The analytic sample was well-balanced across the five establishment size
categories, providing sufficient statistical power for size-stratified analyses:

- 1–4 workers: n = {int(pct_1*N2/100):,} ({pct_1:.1f}%)
- 5–19 workers: n = {int(pct_2*N2/100):,} ({pct_2:.1f}%)
- 20–49 workers: n = {int(pct_3*N2/100):,} ({pct_3:.1f}%)
- 50–99 workers: n = {int(pct_4*N2/100):,} ({pct_4:.1f}%)
- ≥100 workers: n = {int(pct_5*N2/100):,} ({pct_5:.1f}%)

This distribution reflects the underlying Korean establishment population, in
which micro and small enterprises predominate numerically while larger firms
account for a disproportionate share of employment and injury exposure.

---

**Auto-generated from 1_preprocess.ipynb.** Analytic sample SHA-256: `{sha}`.
"""

md_fp = os.path.join(PAPER_DIR, '02_methods_section_2.4-2.6.md')
with open(md_fp, 'w', encoding='utf-8') as f:
    f.write(methods_md)

print(f'\n✓ Saved CSV: {out_fp}')
print(f'✓ Saved Methods (2.4–2.6): {md_fp}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha
Loaded: N=20,262

=== STROBE FLOW ===
Raw:                20,262
- SC missing:            0
After Step 1:       20,262
- Size/Ind miss:         0
Final:              20,262

=== Unadjusted SC Quartile vs Accident Rate ===
  Q1 (SC 3.09): any_acc=7.9%
  Q2 (SC 3.91): any_acc=10.7%
  Q3 (SC 4.41): any_acc=14.6%
  Q4 (SC 4.96): any_acc=15.9%

✓ Saved CSV: /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha/Code_kosha/2_code/output/pre_output/analytic_sample.csv
✓ Saved Methods (2.4–2.6): /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha/Code_kosha/2_code/paper/auto/02_methods_section_2.4-2.6.md
